# Projeto Final: análise das candidaturas da eleição de 2022

**Autores:** Felipe Albanez de Oliveira

## Datasets utilizados
- **Base principal:** consulta_cand_2022_BRASIL.csv
- **Base auxiliar:**  consulta_cand_complementar_2022_BRASIL.csv

## 1. Importando as bibliotecas Pandas e Numpy

In [9]:
# Para facilitar no desenvolvimento, "apelidamos" as bibliotecas no momento da importação
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

## 2. Criando o DataFrame

### 2.1. Erro na primeira tentativa de criação do DataFrame

In [ ]:
"""
caminho note pessoal
pasta_arquivo = "/Users/felipealbanez/Github/CAIXAVERSO_DFF/Projeto_Final_TSE_2022/consulta_cand_2022_BRASIL.csv"

caminho Trabalho
pasta_arquivo = "/Users/cxxxxxx/OneDrive - Caixa Economica Federal/Área de Trabalho/CAIXA Verso/Projeto Final/Base de dados/consulta_cand_2022_BRASIL.csv"

df_original = pd.read_csv(pasta_arquivo)
df_original.head()

erro gerado:
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc7 in position 838: invalid continuation byte

"""

### 2.2. Segunda tentativa - modo "a brasileira"

In [ ]:
""""
sep=';': Os dados usam ponto e vírgula para separar as colunas
decimal=",": números decimais usam vírgula
encoding='latin-1': Evita erros de leitura com acentos e caracteres da língua portuguesa.
"""
# substitua o caminho abaixo pela pasta onde está salvo a base de dados .csv
# não esquecer que o nome do arquivo e a extensão deve estar inclusos

# caminho note pessoal
pasta_arquivo = "/Users/felipealbanez/Github/CAIXAVERSO_DFF/Projeto_Final_TSE_2022/consulta_cand_2022_BRASIL.csv"

# caminho Trabalho
# pasta_arquivo = "/Users/cxxxxxx/OneDrive - Caixa Economica Federal/Área de Trabalho/CAIXA Verso/Projeto Final/Base de dados/consulta_cand_2022_BRASIL.csv"

# cria o dataframe lendo o arquivo .csv
df_original = pd.read_csv(pasta_arquivo, sep=';', decimal=',',encoding='latin-1')

In [ ]:
# vizualisa as 5 primeiras linhas do dataframe
df_original.head()

## 3. Conhecendo o DataFrame

### 3.1. Dimensões e Estrutura do DataFrame

In [ ]:
# shape: informa o número de linhas e colunas
print(f'Shape: {df_original.shape}')
print(f'Quantidade de linhas: {df_original.shape[0]}')
print(f'Quantidade de colunas: {df_original.shape[1]}')

In [ ]:
# info: informa os tipos, faltantes, memória
df_original.info()        

In [ ]:
# describle aula
display(df_original.describe().round(2))
display(df_original.describe(include='object'))

### 3.2 Avaliando as categorizaçãos

In [ ]:
# Mostra os nomes das colunas.
df_original.columns

In [ ]:
# Avaliação de algumas colunas que serão utilizadas
print(df_original['DS_GENERO'].value_counts(dropna=False))
print()
print(df_original['DS_COR_RACA'].value_counts(dropna=False))
print()
print(df_original['DS_GRAU_INSTRUCAO'].value_counts(dropna=False))

## 4. Tratamento de Faltantes (NaN)

### 4.1. Faltantes puros

In [ ]:
# cria um df chamado Faltantes
# são criadas tres colunas para o df 
# isna.sum: percorre a coluna e retorna a soma dos faltantes
# isna.mean: percorre a coluna e retorna a média dos faltantes em relação ao número total de itens na coluna
# nunique: conta quantos valores diferentes cada coluna tem
faltantes = pd.DataFrame({
    'Faltantes': df_original.isna().sum(),
    '%': (df_original.isna().mean() * 100).round(2),
    'distintos': df_original.nunique(),
})

# filta somente as colunas que não possui faltantes
# sort e ascending: ordena pela coluna faltante do maior para o menor
faltantes[faltantes['Faltantes'] > 0].sort_values('Faltantes', ascending=False)

Devido a baixíssima quantidades de faltantes, não é necessário a exclusão da coluna.

### 4.2 Outros tipos de faltantes
Analisando a tabela, verificamos que existe células preenchidas com texto que podem ser considerados 'faltantes'.

In [ ]:
nulos = pd.DataFrame({
    '#NULO': (df_original == '#NULO').sum(),
    '%': ((df_original == '#NULO').mean() * 100).round(2),
})

nulos[nulos['#NULO'] > 0].sort_values('#NULO', ascending=False)

In [ ]:
naodivulgavel = pd.DataFrame({
    'NÃO DIVULGÁVEL': (df_original == 'NÃO DIVULGÁVEL').sum(),
    '%': ((df_original == 'NÃO DIVULGÁVEL').mean() * 100).round(2),
})

naodivulgavel[naodivulgavel['NÃO DIVULGÁVEL'] > 0].sort_values('NÃO DIVULGÁVEL', ascending=False)

### 4.3. Substituindo por NaN
#NULO e NÃO DIVULGAVEL são strings, portanto é necessário transformar em NaN.

In [ ]:
df_original = df_original.replace('#NULO', np.nan)
df_original = df_original.replace('NÃO DIVULGÁVEL', np.nan)

### 4.4. Contagem de Faltantes após o tratamento

In [ ]:
faltantes_atualizado = pd.DataFrame({
    'Faltantes_atualizado': df_original.isna().sum(),
    '%': (df_original.isna().mean() * 100).round(2),
})

# filta somente as colunas que não possui faltantes
# sort e ascending: ordena pela coluna faltante do maior para o menor
faltantes_atualizado[faltantes_atualizado['Faltantes_atualizado'] > 0].sort_values('Faltantes_atualizado', ascending=False)

## 5. Tratamento de Duplicados
      

In [60]:
# verifica se existe linhas com todos os dados duplicados
print('Linhas totalmente duplicadas:', df_original.duplicated().sum())

Linhas totalmente duplicadas: 0


## 6. Tratamento da base auxiliar

Visando atender à exigência de uso do Merge e de três colunas numéricas, utilizou-se a tabela consulta_cand_2022_BRASIL, selecionando os campos NR_IDADE_DATA_POSSE e VR_DESPESA_MAX_CAMPANHA.

**Observação** = VR_DESPESA_MAX_CAMPANHA é o máximo que o candidato pode utilizar. Portanto, não quer dizer que foi utilizado tudo.

### 6.1. Lendo a base auxiliar

In [ ]:
# substitua o caminho abaixo pela pasta onde está salvo a base de dados .csv
# não esquecer que o nome do arquivo e a extensão deve estar inclusos

pasta_arquivo = "/Users/felipealbanez/Github/CAIXAVERSO_DFF/Projeto_Final_TSE_2022/consulta_cand_complementar_2022_BRASIL.csv"

# cria o dataframe lendo o arquivo .csv
auxiliar = pd.read_csv(pasta_arquivo, sep=';', decimal=',',encoding='latin-1')

# Seleciona a chave e apenas as duas colunas necessárias
# o SQ_CANDIDATO é número sequencial identificador único do candidato dentro da base do TSE
# funciona como um "id"
auxiliar = auxiliar[['SQ_CANDIDATO','NR_IDADE_DATA_POSSE', 'VR_DESPESA_MAX_CAMPANHA']].copy()

"""
# Converte as colunas para número.
complementar['NR_IDADE_DATA_POSSE'] = pd.to_numeric(
    complementar['NR_IDADE_DATA_POSSE'],
    errors='coerce'
)
complementar['VR_DESPESA_MAX_CAMPANHA'] = pd.to_numeric(
    complementar['VR_DESPESA_MAX_CAMPANHA'],
    errors='coerce'
)

# Remove chaves repetidas na auxiliar, mantendo a primeira.
complementar = complementar.drop_duplicates(
    subset='SQ_CANDIDATO',
    keep='first'
).reset_index(drop=True)
"""

auxiliar.head()

,SQ_CANDIDATO,NR_IDADE_DATA_POSSE,VR_DESPESA_MAX_CAMPANHA
0,100001613736,49.0,1270629.01
1,160001597835,45.0,3176572.53
2,160001621917,41.0,3176572.53
3,110001643976,39.0,1270629.01
4,110001618165,46.0,1270629.01


In [69]:
# Identifica idades inválidas.
idades_invalidas = auxiliar[
    (auxiliar['NR_IDADE_DATA_POSSE'] < 18) |
    (auxiliar['NR_IDADE_DATA_POSSE'] > 100)
]
print('Idades inválidas:', len(idades_invalidas))

"""
# Substitui idades inválidas por NaN.
complementar.loc[
    (complementar['NR_IDADE_DATA_POSSE'] < 18) |
    (complementar['NR_IDADE_DATA_POSSE'] > 100),
    'NR_IDADE_DATA_POSSE'
] = np.nan
"""

# Identifica limites de despesas negativos.
despesas_invalidas = auxiliar[
    auxiliar['VR_DESPESA_MAX_CAMPANHA'] < 0
]
print('Despesas negativas:', len(despesas_invalidas))

"""
# Substitui despesas negativas por NaN.
complementar.loc[
    complementar['VR_DESPESA_MAX_CAMPANHA'] < 0,
    'VR_DESPESA_MAX_CAMPANHA'
] = np.nan
"""

Idades inválidas: 0
Despesas negativas: 859


"\n# Substitui despesas negativas por NaN.\ncomplementar.loc[\n    complementar['VR_DESPESA_MAX_CAMPANHA'] < 0,\n    'VR_DESPESA_MAX_CAMPANHA'\n] = np.nan\n"

## 7. Junçao das bases com Merge

In [76]:
# merge com a chave SQ_CANDIDATO
df_merge = df_original.merge(auxiliar, on='SQ_CANDIDATO',how='left')

# dimensoes de antes e depois
print("Shape antes do merge:", df_original.shape)
print("Shape depois do merge:", df_merge.shape)

Shape antes do merge: (29322, 50)
Shape depois do merge: (29426, 52)
